<a href="https://colab.research.google.com/github/masonness18/module-2-/blob/main/Assignment_2_edited.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
pip install --upgrade gradio

In [15]:
from google import genai
from google.colab import userdata
import os
import datetime

# Retrieve the actual API key value from Google Colab Secrets with a fallback
try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    try:
        api_key = userdata.get('MASON_API_KEY')
    except Exception:
        api_key = os.environ.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)

def totals_by_category(transactions):
    spending={}
    for t in transactions:
        if t['category'] not in spending:
            spending[t['category']] = 0
        spending[t['category']] += t['amount']
    return spending

def calculate_budget_status(total_budget,spent_so_far,trip_start,trip_end,today):
    total_days=(trip_end-trip_start).days
    days_left=total_days-(today-trip_start).days
    actual_pct=(spent_so_far/total_budget)*100
    pct_left=100-actual_pct
    expected_pct=((today-trip_start).days/total_days)*100
    gap=actual_pct-expected_pct

    if gap <= 10:
        status="On track"
    else:
        status="Overspending"

    return {
        "status":status,
        "pct_actual":actual_pct,
        "gap":gap,
        "total_days":total_days,
        "days_left":days_left,
        "total_budget":total_budget,
        "spent_so_far":spent_so_far,
        "pct_left":pct_left
    }

def ask_with_data(chat,question,transactions,total_budget,trip_start,trip_end,today):
    spent=sum(t['amount'] for t in transactions)
    spending_by_category=totals_by_category(transactions)
    budget_status=calculate_budget_status(total_budget,spent,trip_start,trip_end,today)

    full_message="spending by category:" + str(spending_by_category) + "\n" + str(budget_status)
    full_message+="budget status:" + str(budget_status) + "\n"

    response=chat.send_message(question + full_message)
    return response.text

EXCHANGE_RATES={
    'AUD': 1.0,
    'IDR': 12626.0,
}

def convert_currency(amount, from_currency, to_currency):
    if from_currency == to_currency:
        return amount

    try:
        amount = float(amount)
    except ValueError:
        return "Invalid amount"

    if amount <= 0:
        return "Invalid amount"

    if from_currency not in EXCHANGE_RATES or to_currency not in EXCHANGE_RATES:
        return "Invalid currency"

    convert_amount = amount / EXCHANGE_RATES[from_currency] * EXCHANGE_RATES[to_currency]
    return convert_amount

def run_app(chat,total_budget,trip_start,trip_end,today):
    transactions=[]
    running=True
    while running:
        print("\n1. Ask a question")
        print("2. Log an expense")
        print("3. Quit")

        choice=input("Enter your choice (1-3):").strip()

        if choice == "1":
            question=input("Enter your question:")
            response=ask_with_data(chat,question,transactions,total_budget,trip_start,trip_end,today)
            print(response)
        elif choice =="2":
            category=input("Enter the category:").strip()
            while True:
                raw_amount=input("Enter the amount:").strip()

                for char in ["$", ","]:
                    raw_amount = raw_amount.replace(char, "")
                try:
                    amount=float(raw_amount)
                    break
                except ValueError:
                    print("Invalid input. Please enter a valid number (e.g. 15 or 12.50).")
            transactions.append({"category":category,"amount":amount})
            print(f"Logged ${amount:.2f} to {category}.")
        elif choice == "3":
            running=False

persona_setup = """
You are the AI assitant inside a travel budget tracking app. Your persona is a supportive but strict travel buddy.

The user is on a 2 month moped surf trip through Indonesia with a $10,000 budget (excluding flights and the moped). On their last trip (8 months in Europe) they struggled to keep to their budget and had to dip into savings and find work.

You will be given the user's REAL, CURRENT spending data with every message, give a Day-to-Day breakdown by category (food, drinks, fuel and miscellaneous) and overall budget status. Your analysis and advice must be grounded in those actual numbers evertime, not generic advice. If the numbers change, your response should change to match them.
"""

chat=client.chats.create(model="gemini-3.5-flash")
chat.send_message(persona_setup)

run_app(chat,total_budget=10000,trip_start=datetime.date(2027,7,1),trip_end=datetime.date(2027,9,1),today=datetime.date.today())


1. Ask a question
2. Log an expense
3. Quit
Enter your choice (1-3):2
Enter the category:food
Enter the amount:$15
Logged $15.00 to food.

1. Ask a question
2. Log an expense
3. Quit
Enter your choice (1-3):3


In [14]:
import gradio as gr
import pandas as pd

# Initialize state inside the notebook runtime
active_transactions = []
total_budget = 10000
trip_start = datetime.date(2027, 7, 1)
trip_end = datetime.date(2027, 9, 1)
today = datetime.date.today()

def get_budget_summary():
    spent = sum(t['amount'] for t in active_transactions)
    status_info = calculate_budget_status(total_budget, spent, trip_start, trip_end, today)

    summary_md = f"""
    ### Current Budget Status: **{status_info['status']}**
    * **Total Budget:** ${status_info['total_budget']:.2f}
    * **Total Spent:** ${status_info['spent_so_far']:.2f}
    * **Budget Remaining:** ${status_info['total_budget'] - status_info['spent_so_far']:.2f} ({status_info['pct_left']:.1f}%)
    * **Days Left of Trip:** {status_info['days_left']} days (out of {status_info['total_days']})
    * **Pace Deviation (Gap):** {status_info['gap']:.1f}%
    """
    return summary_md, pd.DataFrame(active_transactions) if active_transactions else pd.DataFrame(columns=["category", "amount"])

def log_expense_gradio(category, amount):
    if not category.strip():
        return "Please enter a valid category.", *get_budget_summary()
    try:
        # Strip dollar signs if users type them
        amount_clean = str(amount).replace('$', '').replace(',', '').strip()
        amt = float(amount_clean)
    except ValueError:
        return "Please enter a numeric amount.", *get_budget_summary()

    active_transactions.append({"category": category.strip(), "amount": amt})
    summary, df = get_budget_summary()
    return f"Logged ${amt:.2f} to {category}!", summary, df

def chat_gradio(message, history):
    # Prepare current context payload
    spent = sum(t['amount'] for t in active_transactions)
    spending_by_category = totals_by_category(active_transactions)
    budget_status = calculate_budget_status(total_budget, spent, trip_start, trip_end, today)

    full_message = f"{message}\n\n[Current Data - Spending by Category: {spending_by_category}, Budget Status: {budget_status}]"
    response = chat.send_message(full_message)
    return response.text

with gr.Blocks() as demo:
    gr.Markdown("# 🌴 Moped Surf Budget Tracker & AI Buddy")

    with gr.Tab("💸 Log Expenses"):
        with gr.Row():
            with gr.Column():
                category_input = gr.Textbox(label="Category (e.g., Food, Drinks, Fuel)", placeholder="Enter category...")
                amount_input = gr.Textbox(label="Amount ($)", placeholder="e.g. 15.50")
                log_button = gr.Button("Log Expense", variant="primary")
                status_feedback = gr.Markdown()

            with gr.Column():
                budget_status_display = gr.Markdown("### Add some expenses to view status summary!")
                transactions_table = gr.Dataframe(headers=["category", "amount"], datatype=["str", "number"], label="Logged Transactions")

        log_button.click(
            fn=log_expense_gradio,
            inputs=[category_input, amount_input],
            outputs=[status_feedback, budget_status_display, transactions_table]
        )

    with gr.Tab("💬 Chat with AI Buddy"):
        gr.Markdown("Ask your strict travel buddy anything about your current budget and spending pattern.")
        gr.ChatInterface(
            fn=chat_gradio
        )

demo.launch(theme=gr.themes.Soft(), debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1f242ece7190689dc9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1f242ece7190689dc9.gradio.live
